In [1]:
import torch

import random
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 473

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")



Training on: cuda


# Import CIFAR-10 (in-distribution) dataset

In [2]:
# Import the batches here

# Note to self: These are hardcoded and apparently come from the actual CIFAR-10 dataset.
#	It's just to streamline the calculations instead of manually having to do it. I think the
#	tuple values are a (R,G,B) for the color channels.
mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

# Two versions of the training split are needed so validation has no augmentation.
train_data = datasets.CIFAR10(
    root="./data",
    train=True,
    download=False,
    transform=train_transform,
)
eval_data = datasets.CIFAR10(
    root="./data",
    train=True,
    download=False,
    transform=eval_transform,
)
test_data = datasets.CIFAR10(
    root="./data",
    train=False,
    download=False,
    transform=eval_transform,
)

print(f"Training images available: {len(train_data)}")
print(f"Test images: {len(test_data)}")
print(f"Classes: {train_data.classes}")

RuntimeError: Dataset not found or corrupted. You can use download=True to download it

In [ ]:
split_generator = torch.Generator().manual_seed(SEED)
indices = torch.randperm(len(train_data), generator=split_generator)

val_indices = indices[:5000].tolist()
train_indices = indices[5000:].tolist()

train_loader = DataLoader(
    Subset(train_data, train_indices),
    batch_size=128,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
    num_workers=0,
)

val_loader = DataLoader(
    Subset(eval_data, val_indices),
    batch_size=256,
    shuffle=False,
    num_workers=0,
)

test_loader = DataLoader(
    test_data,
    batch_size=256,
    shuffle=False,
    num_workers=0,
)

# Import SVHN (out-of-distribution) dataset

In [ ]:
# Import the batches here (street view house numbers)

# Import ResNet-20 model

In [ ]:
# This hub repository is structured correctly and supports pretrained=False
model = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar10_resnet20", pretrained=False)
model = model.to(device)  # move to the GPU

print(f"Total parameters: {sum(p.numel() for p in model.parameters())}")

Total parameters: 272474


Using cache found in C:\Users\ellio/.cache\torch\hub\chenyaofo_pytorch-cifar-models_master


In [ ]:
## Set up training loss
criterion = torch.nn.CrossEntropyLoss()

epochs = 40 # number of iterations through the training batch
optimizer = torch.optim.SGD(
	model.parameters(),  # use the parameters (?) obtained by the model
	lr=0.1, # learning rate.
	momentum=0.9, # m
	weight_decay = 0.0005
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

In [ ]:
# Train the model
def accuracy(loader):

	model.eval()
	correct = 0
	total = 0

	with torch.no_grad():  # why no gradient?
		for images, labels in loader:
			images = images.to(device)
			labels = labels.to(device)

			## Perform the softmax on the images
			predictions = model(images).argmax(dim=1)
			correct += (predictions == labels).sum().item()
			total += labels.size(0)

		return correct / total

output_path = Path("output/cifar10_resnet_20.pt")
output_path.parent.mkdir(parents=True, exist_ok=True)
best_val_accuracy = 0.0

for epoch in range(epochs):
	model.train()
	total_loss = 0.0
	seen = 0

	for images, labels in train_loader:
		images = images.to(device)
		labels = labels.to(device)

		optimizer.zero_grad() # set the gradients to zero for init
		logits = model(images)  # note to self: logits are the pre-activated output of the final layer
		loss = criterion(logits, labels)
		loss.backward()

		optimizer.step()

		total_loss += loss.item() * labels.size(0)
		seen += labels.size(0)

	scheduler.step()
	val_accuracy = accuracy(val_loader)
	average_loss = total_loss / seen

	print(
		f"Epoch {epoch} / {epochs}" f"loss={average_loss}" f"val_accuracy={val_accuracy}"
	)

	if val_accuracy > best_val_accuracy:
		best_val_accuracy = val_accuracy
		torch.save({
			"model_name": "cifar-10-resnet-20",
			"state_dict": model.state_dict(),
			"epoch": epoch + 1,
			"val_accuracy": val_accuracy,
			"seed": SEED,
			"classes": train_data.classes,
			"mean": mean,
			"std": std
		}, output_path)


print(f"Best validation accuracy: {best_val_accuracy}")
print(f"Output saved to {output_path}")


# Note: get NLL and ECE


Epoch 0 / 40loss=1.628435597652859val_accuracy=0.4454
Epoch 1 / 40loss=1.1220026180055407val_accuracy=0.5856
Epoch 2 / 40loss=0.9193680265744527val_accuracy=0.6322
Epoch 3 / 40loss=0.7756767494625515val_accuracy=0.7168
Epoch 4 / 40loss=0.7028818111313714val_accuracy=0.7168
Epoch 5 / 40loss=0.6615938615375095val_accuracy=0.6916
Epoch 6 / 40loss=0.6192299564997356val_accuracy=0.7332
Epoch 7 / 40loss=0.5962546144697402val_accuracy=0.76
Epoch 8 / 40loss=0.5792484313170115val_accuracy=0.7344
Epoch 9 / 40loss=0.5596166990915934val_accuracy=0.7246
Epoch 10 / 40loss=0.5303635505358378val_accuracy=0.747
Epoch 11 / 40loss=0.5235656554751926val_accuracy=0.7466
Epoch 12 / 40loss=0.49980922433005437val_accuracy=0.7094
Epoch 13 / 40loss=0.49256146047380234val_accuracy=0.7566
Epoch 14 / 40loss=0.4705533327844408val_accuracy=0.7598
Epoch 15 / 40loss=0.4595463183085124val_accuracy=0.799
Epoch 16 / 40loss=0.44635510370466447val_accuracy=0.807
Epoch 17 / 40loss=0.4312521988021003val_accuracy=0.764
Epoch 